# NUMPY MASTERY — CHAPTERS 62–65

```
NUMPY MASTERY
Current Chapters: 62 (NumPy Project Structure), 63 (Testing NumPy Code),
                  64 (Numerical Edge Cases), 65 (Advanced NumPy)
Progress:         65 / 75 chapters
Completed:        Ch1-61
Remaining:        10 chapters
```

How to organize a real NumPy project (62), how to test it properly (63), a systematic tour of
every edge case a numerical function needs to survive (64), and the advanced tools —
`einsum` gets its own full chapter next — worth knowing exist even if used rarely (65).


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 62 — NumPy Project Structure

## Concept
A notebook is for exploration. A project is for reuse, testing, and collaboration. The
standard structure below is how numerical/ML Python projects are organized in industry —
recognizing it is as important as being able to write the code inside it.

```
numpy-ml-project/
│
├── data/                    # raw and processed datasets (often git-ignored if large)
│   ├── raw/
│   └── processed/
│
├── notebooks/                # exploratory work -- Chapters 1-61's style of iteration
│   └── 01_eda.ipynb
│
├── src/                       # the REAL, tested, reusable code -- what notebooks import FROM
│   ├── __init__.py
│   ├── preprocessing.py         # Chapter 57's CustomerDataProcessor-style classes
│   ├── numerical_utils.py         # Chapter 27/28/29's stability/memory helpers
│   ├── features.py                  # Chapter 35's feature engineering functions
│   ├── models.py                      # Chapter 38-46's from-scratch implementations
│   └── evaluation.py                    # Chapter 13/14's metrics (R^2, accuracy, etc.)
│
├── tests/                        # Chapter 63's test suite, one file per src/ module
│   ├── test_preprocessing.py
│   └── test_models.py
│
├── reports/                        # generated figures, model performance summaries
│
├── requirements.txt                  # pinned dependencies (numpy==2.4.4, etc.)
│
└── README.md
```

## Why the notebooks/src split matters
Notebooks execute cells in whatever order you happened to run them — this is fantastic for
exploration and fatal for correctness guarantees. Code in `src/` is imported as ordinary Python
modules, executes top-to-bottom deterministically, and can be unit tested (Chapter 63) — the
exact discipline this entire course has already been applying every time a build script wrote a
standalone `.py` file, ran it, and verified the output before trusting it.

## How notebook experimentation becomes production code
```python
# Stage 1 (in a notebook): explore inline
X_clean = X[np.isfinite(X).all(axis=1)]
X_scaled = (X_clean - X_clean.mean(axis=0)) / X_clean.std(axis=0)

# Stage 2 (promoted to src/preprocessing.py): wrapped as a tested, documented function
# -- literally Chapter 56's CustomerDataProcessor pattern
def clean_and_scale(X: np.ndarray) -> np.ndarray:
    """Drop non-finite rows, then z-score standardize. See tests/test_preprocessing.py."""
    X = np.asarray(X)
    X_clean = X[np.isfinite(X).all(axis=1)]
    mean, std = X_clean.mean(axis=0), X_clean.std(axis=0)
    std = np.where(std < 1e-12, 1.0, std)
    return (X_clean - mean) / std

# Stage 3 (back in the notebook): import and use, don't redefine
# from src.preprocessing import clean_and_scale
# X_scaled = clean_and_scale(X)
```

### Interview Insight
> "Why keep exploratory notebook code separate from `src/` code in an ML project?"
Notebooks execute non-linearly (cells can be re-run out of order, leaving stale variable state)
and aren't easily unit-tested; `src/` modules execute deterministically top-to-bottom and can be
imported, tested, and reused across multiple notebooks/scripts/services without duplicating
logic — the separation is what makes a codebase maintainable past the "one person, one
notebook" stage.


---
# CHAPTER 63 — Testing NumPy Code

## Concept
`==` is almost never the right assertion for numerical code (Chapter 27's floating-point
lesson, now formalized as a testing discipline). `np.testing` provides purpose-built assertions
for exact equality, tolerance-based equality, shape checking, and dtype checking.


In [ ]:
def standardize(X, axis=0):
    mean = X.mean(axis=axis, keepdims=True)
    std = X.std(axis=axis, keepdims=True)
    std = np.where(std < 1e-12, 1.0, std)
    return (X - mean) / std


# --- np.testing assertions, and what each is actually for ---
a = np.array([1.0, 2.0, 3.0])
b = np.array([1.0, 2.0, 3.0])
c = np.array([1.0000001, 2.0, 3.0])   # tiny floating point difference

print("Testing exact equality (np.testing.assert_array_equal):")
np.testing.assert_array_equal(a, b)   # passes silently -- no output means success
print("  a == b: PASSED (identical values)")

try:
    np.testing.assert_array_equal(a, c)
except AssertionError as e:
    print("  a == c: FAILED as expected (even a tiny difference fails exact equality)")

print("\nTesting tolerance-based equality (np.testing.assert_allclose):")
np.testing.assert_allclose(a, c, rtol=1e-5)   # passes -- within tolerance
print("  a ~= c: PASSED (within rtol=1e-5)")

print("\nTesting shape (a common, easy-to-forget check):")
result = standardize(np.random.default_rng(0).normal(size=(10, 5)))
assert result.shape == (10, 5), f"Expected (10,5), got {result.shape}"
print("  standardize output shape: PASSED")


### Expected Output (verified on NumPy 2.4.4)
```
Testing exact equality (np.testing.assert_array_equal):
  a == b: PASSED (identical values)
  a == c: FAILED as expected (even a tiny difference fails exact equality)

Testing tolerance-based equality (np.testing.assert_allclose):
  a ~= c: PASSED (within rtol=1e-5)

Testing shape (a common, easy-to-forget check):
  standardize output shape: PASSED
```

### A real, structured test suite for `standardize` — the standard set of cases to cover


In [ ]:
def test_standardize_basic_correctness():
    X = np.array([[1., 2.], [3., 4.], [5., 6.]])
    result = standardize(X)
    np.testing.assert_allclose(result.mean(axis=0), [0, 0], atol=1e-10)
    np.testing.assert_allclose(result.std(axis=0), [1, 1], atol=1e-10)

def test_standardize_constant_column_no_nan():
    """Edge case (Chapter 36): a constant column must not produce NaN/Inf."""
    X = np.array([[1., 5.], [2., 5.], [3., 5.]])
    result = standardize(X)
    assert not np.isnan(result).any(), "constant column produced NaN"
    assert not np.isinf(result).any(), "constant column produced Inf"
    np.testing.assert_allclose(result[:, 1], [0, 0, 0])   # constant column -> all zeros after centering

def test_standardize_shape_preserved():
    X = np.random.default_rng(0).normal(size=(50, 8))
    result = standardize(X)
    assert result.shape == X.shape

def test_standardize_does_not_mutate_input():
    """Chapter 8/56: standardize must not silently mutate the caller's array."""
    X = np.array([[1., 2.], [3., 4.]])
    X_original = X.copy()
    standardize(X)
    np.testing.assert_array_equal(X, X_original)

def test_standardize_single_row():
    """Edge case (Chapter 64 preview): a single row has zero variance per column."""
    X = np.array([[1., 2., 3.]])
    result = standardize(X)
    assert not np.isnan(result).any()


tests = [
    test_standardize_basic_correctness,
    test_standardize_constant_column_no_nan,
    test_standardize_shape_preserved,
    test_standardize_does_not_mutate_input,
    test_standardize_single_row,
]

for test_fn in tests:
    try:
        test_fn()
        print(f"PASS: {test_fn.__name__}")
    except AssertionError as e:
        print(f"FAIL: {test_fn.__name__} -- {e}")


### Expected Output (verified on NumPy 2.4.4)
```
PASS: test_standardize_basic_correctness
PASS: test_standardize_constant_column_no_nan
PASS: test_standardize_shape_preserved
PASS: test_standardize_does_not_mutate_input
PASS: test_standardize_single_row
```

### Why `test_standardize_does_not_mutate_input` is not a paranoid, unnecessary test
This directly tests for Chapter 8/50's most common class of NumPy bug — silent input mutation.
Without this test explicitly asserting `X` is unchanged after the call, a future refactor could
reintroduce an in-place operation (`X -= mean` instead of `X = X - mean`) and every *other* test
might still pass (since they check the *return value*, not whether the input was corrupted along
the way) while silently breaking any caller who reuses `X` afterward. **Testing for the absence
of mutation is not optional for any function documented as "does not modify its input."**

### `assert_allclose` parameters worth knowing precisely
`rtol` (relative tolerance) and `atol` (absolute tolerance) combine as: values are considered
close if `|actual - desired| <= atol + rtol * |desired|`. Use a larger `atol` when comparing
values near zero (where relative tolerance alone is too strict — `rtol` of anything times
nearly-zero is nearly-zero, making the check nearly-exact by accident); use `rtol` for
general-magnitude comparisons, exactly the reasoning behind Chapter 27's `np.isclose` defaults.

### Interview Insight
> "Why is `assert result == expected` almost always wrong for testing numerical NumPy code?"
Floating-point arithmetic accumulates tiny representation/rounding errors (Chapter 27) even for
mathematically-equivalent computations — exact equality is too strict a bar for almost any
nontrivial numerical result. `np.testing.assert_allclose` (or `np.isclose`) with sensible
`rtol`/`atol` is the correct standard, reserving exact equality checks for genuinely
integer/discrete outputs (shapes, counts, boolean masks) where floating-point drift can't occur.


---
# CHAPTER 64 — Numerical Edge Cases

## Concept
A systematic tour of the inputs that break numerical functions in practice — most of which
this course has already touched on individually. This chapter is the consolidated checklist.

Note: the probes below wrap risky operations in `np.errstate(...)` purely to keep this
notebook's output clean for demonstration — in real code you should generally **let these
warnings surface** (or explicitly configure `np.seterr` at the application boundary) rather than
silently suppressing them everywhere, since the warning is often the only signal something
unexpected happened.


In [ ]:
def probe_edge_case(name, fn):
    try:
        result = fn()
        print(f"{name:30s}: OK -> {result}")
    except Exception as e:
        print(f"{name:30s}: RAISED {type(e).__name__}: {e}")


print("--- Empty and tiny arrays ---")
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', category=RuntimeWarning)
    probe_edge_case("empty array .mean()", lambda: np.array([]).mean())
probe_edge_case("single-element .std()", lambda: np.array([5.0]).std())
probe_edge_case("empty array reshape(0,3)", lambda: np.array([]).reshape(0, 3).shape)

print("\n--- Zero variance ---")
probe_edge_case("constant array std()", lambda: np.array([7., 7., 7.]).std())
with np.errstate(invalid='ignore'):
    probe_edge_case("z-score of constant array", lambda: ((np.array([7.,7.,7.]) - 7.0) / np.array([7.,7.,7.]).std()))

print("\n--- Division / log edge cases ---")
with np.errstate(divide='ignore', invalid='ignore'):
    probe_edge_case("1.0 / 0.0", lambda: np.array(1.0) / np.array(0.0))
    probe_edge_case("0.0 / 0.0", lambda: np.array(0.0) / np.array(0.0))
    probe_edge_case("log(0)", lambda: np.log(np.array(0.0)))
    probe_edge_case("log(-1)", lambda: np.log(np.array(-1.0)))

print("\n--- Integer overflow ---")
with np.errstate(over='ignore'):
    probe_edge_case("int8: 127 + 1", lambda: np.int8(127) + np.int8(1))
    probe_edge_case("uint8: 0 - 1", lambda: np.uint8(0) - np.uint8(1))

print("\n--- Shape mismatches ---")
probe_edge_case("wrong dimension count", lambda: np.array([1,2,3]) @ np.array([[1,2],[3,4],[5,6]]))


### Expected Output (verified on NumPy 2.4.4)
```
--- Empty and tiny arrays ---
empty array .mean()          : OK -> nan
single-element .std()        : OK -> 0.0
empty array reshape(0,3)     : OK -> (0, 3)

--- Zero variance ---
constant array std()         : OK -> 0.0
z-score of constant array    : OK -> [nan nan nan]

--- Division / log edge cases ---
1.0 / 0.0                    : OK -> inf
0.0 / 0.0                    : OK -> nan
log(0)                       : OK -> -inf
log(-1)                      : OK -> nan

--- Integer overflow ---
int8: 127 + 1                : OK -> -128
uint8: 0 - 1                 : OK -> 255

--- Shape mismatches ---
wrong dimension count        : OK -> [22 28]
```

### Reading this table — the recurring pattern
**Every single one of these "ran without error."** Not one raised an exception — `nan`,
`inf`, `-128`, `255`, and silently-questionable-but-technically-valid shape results all came
back as ordinary-looking successful return values. This is the single most important fact
about NumPy edge-case behavior: **NumPy overwhelmingly prefers silent, well-defined-but-
surprising results over exceptions**, for performance reasons (checking every element for
validity on every operation would be prohibitively expensive at scale). This is *why* Chapters
15, 4, and this chapter's own probing pattern (`np.isnan`/`np.isinf`/`np.isfinite` checks)
aren't optional defensive habits — they're the **only** mechanism that will ever tell you one
of these silent edge cases occurred, since NumPy itself won't.

### The defensive checklist this implies for any production numerical function
```python
def robust_function(X):
    X = np.asarray(X)
    if X.size == 0:
        raise ValueError("Input array is empty")          # Chapter 64: empty -> often NaN, not always sensible
    if not np.isfinite(X).all():
        raise ValueError("Input contains NaN or infinite values")   # Chapter 15
    if X.ndim != 2:
        raise ValueError(f"Expected 2D input, got shape {X.shape}")  # Chapter 3
    # ... only now proceed with the actual computation, on GUARANTEED-clean input
```

### Interview Insight
> "Does NumPy raise exceptions for invalid numerical operations like division by zero?"
No — by default, NumPy computes IEEE-754-defined results (`inf`, `-inf`, `nan`) and emits a
`RuntimeWarning`, not an exception, for operations like division by zero or `log` of a
non-positive number. Integer overflow is even quieter — it wraps around with **no warning at
all**. This is a deliberate performance trade-off, and it's exactly why explicit
`isnan`/`isinf`/`isfinite`/range checks are necessary at the boundaries of production numerical
code, rather than relying on exceptions to surface problems.


---
# CHAPTER 65 — Advanced NumPy

## Concept
A tour of less-common-but-genuinely-useful tools, each shown for its actual practical value —
not included merely because it exists. `einsum` gets a dedicated deep dive in Chapter 66;
here's everything else.


In [ ]:
# --- Structured arrays: heterogeneous, named fields, all in ONE ndarray ---
dt = np.dtype([('name', 'U10'), ('age', 'i4'), ('score', 'f8')])
people = np.array([('Alice', 30, 92.5), ('Bob', 25, 88.1), ('Carol', 35, 95.3)], dtype=dt)

print("Structured array:\n", people)
print("\nAccess by field name:", people['name'])
print("Ages > 28:", people[people['age'] > 28]['name'])
print("dtype:", people.dtype)


### Expected Output (verified on NumPy 2.4.4)
```
Structured array:
 [('Alice', 30, 92.5) ('Bob', 25, 88.1) ('Carol', 35, 95.3)]

Access by field name: ['Alice' 'Bob' 'Carol']
Ages > 28: ['Alice' 'Carol']
dtype: [('name', '<U10'), ('age', '<i4'), ('score', '<f8')]
```

### When structured arrays are actually worth using
When you need heterogeneous, named-field data but want to stay in pure NumPy rather than reach
for pandas (e.g., a lightweight embedded system, or interop with a C struct / binary file
format that has a fixed named layout) — otherwise, pandas is almost always the better tool for
this exact use case (Chapter 54), and structured arrays are worth recognizing more than reaching
for by default.


In [ ]:
# --- np.meshgrid: coordinate grids for vectorized 2D/3D function evaluation ---
x = np.linspace(-2, 2, 5)
y = np.linspace(-1, 1, 3)
X, Y = np.meshgrid(x, y)   # X, Y are both (3, 5) -- every combination of x and y

print("x:", x)
print("y:", y)
print("X (shape {}):\n{}".format(X.shape, X))
print("Y (shape {}):\n{}".format(Y.shape, Y))

Z = X**2 + Y**2   # evaluate a function over the ENTIRE grid at once, no nested loop
print("\nZ = X^2 + Y^2 (shape {}):\n{}".format(Z.shape, np.round(Z, 2)))


### Expected Output (verified on NumPy 2.4.4)
```
x: [-2. -1.  0.  1.  2.]
y: [-1.  0.  1.]
X (shape (3, 5)):
[[-2. -1.  0.  1.  2.]
 [-2. -1.  0.  1.  2.]
 [-2. -1.  0.  1.  2.]]
Y (shape (3, 5)):
[[-1. -1. -1. -1. -1.]
 [ 0.  0.  0.  0.  0.]
 [ 1.  1.  1.  1.  1.]]

Z = X^2 + Y^2 (shape (3, 5)):
[[5. 2. 1. 2. 5.]
 [4. 1. 0. 1. 4.]
 [5. 2. 1. 2. 5.]]
```

### Real-world use
Evaluating a function over a 2D parameter grid for visualization (contour plots), simulating a
spatial field (e.g., temperature or pressure over a physical grid), or evaluating a loss
function's surface over a grid of hyperparameter values for a diagnostic plot — anywhere you'd
otherwise write nested loops over two ranges, `meshgrid` + vectorized arithmetic replaces them
in the same spirit as every "avoid the loop" lesson in this course.

### `np.vectorize` — a documented trap (Chapter 30 callback), and when it's still defensible


In [ ]:
# np.vectorize is NOT true vectorization (Chapter 30's lesson) -- but it's still occasionally
# the RIGHT tool: wrapping a genuinely complex, branchy Python function for BROADCASTING
# convenience, when correctness/readability matters more than raw performance for a small input.

def complex_business_rule(x):
    """A deliberately branchy function that doesn't vectorize naturally with array ops."""
    if x < 0:
        return "invalid"
    elif x < 10:
        return "low"
    elif x < 100:
        return "medium"
    else:
        return "high"

vectorized_rule = np.vectorize(complex_business_rule)
values = np.array([-5, 3, 50, 200])
print("np.vectorize applied to a branchy function:", vectorized_rule(values))
print("(This is legitimate: readability/correctness for a genuinely branchy small-scale rule,")
print(" not a performance optimization -- if this ran on millions of values, rewrite it with")
print(" np.select or np.piecewise instead, per Chapter 30's warning.)")

# The Chapter 30-preferred vectorized alternative for THIS specific case:
def business_rule_vectorized(x):
    conditions = [x < 0, x < 10, x < 100]
    choices = ["invalid", "low", "medium"]
    return np.select(conditions, choices, default="high")   # true vectorized, compiled dispatch

print("\nnp.select alternative (genuinely vectorized):", business_rule_vectorized(values))


### Expected Output (verified on NumPy 2.4.4)
```
np.vectorize applied to a branchy function: ['invalid' 'low' 'medium' 'high']
(This is legitimate: readability/correctness for a genuinely branchy small-scale rule,
 not a performance optimization -- if this ran on millions of values, rewrite it with
 np.select or np.piecewise instead, per Chapter 30's warning.)

np.select alternative (genuinely vectorized): ['invalid' 'low' 'medium' 'high']
```

### `np.select` — genuinely vectorized multi-condition branching
`np.select(conditions, choices, default)` evaluates all conditions and choices as arrays,
picking the first matching condition per element via compiled dispatch (no Python-level
per-element function call, unlike `np.vectorize`) — this is the correct tool when a branchy
rule needs to run at real scale, while `np.vectorize` remains a legitimate (if
performance-costly) choice for small-scale convenience or genuinely irregular logic that
resists array-based expression entirely.

### Interview Insight
> "When, if ever, is `np.vectorize` an acceptable choice, given Chapter 30 showed it's ~100x+
> slower than true vectorization?"
When the input is small enough that the performance gap doesn't matter in absolute terms, and
the underlying logic is genuinely branchy/irregular in a way that's awkward to express with
`np.where`/`np.select`/boolean masking — `np.vectorize` then buys broadcasting *convenience*
(automatic shape handling) at a real but sometimes acceptable performance cost. For anything
running at meaningful scale, `np.select`, `np.piecewise`, or restructuring the logic into
boolean-masked array operations is the correct choice instead.


---
## Chapters 62–65 Summary

- **Project structure**: `notebooks/` for exploration, `src/` for tested/reusable/deterministic
  code, `tests/` mirroring `src/` — the discipline this course's own build-verify-fix cycle has
  modeled throughout, now named explicitly as a project convention.
- **Testing**: `np.testing.assert_allclose` (not `==`) for floating-point results;
  `assert_array_equal` for genuinely exact/discrete outputs; always include a
  does-not-mutate-input test for any function documented as non-mutating — verified live above
  that all 5 example test cases pass.
- **Numerical edge cases**: the unifying fact is that NumPy almost never raises exceptions for
  invalid-but-well-defined operations (division by zero, log of zero/negative, integer overflow,
  empty-array statistics) — every single probed case above returned a silent, valid-looking
  result. Defensive `isfinite`/shape/size checks at function boundaries are the only real
  safeguard, since NumPy itself won't warn loudly.
- **Advanced tools**: structured arrays for lightweight named-field data (usually pandas'
  territory instead); `meshgrid` for vectorized evaluation over 2D/3D grids; `np.vectorize` as a
  legitimate-but-costly convenience tool for small/branchy logic, with `np.select` as the
  genuinely-vectorized alternative when scale matters.

---

